# Training Tasks Notebook

In [42]:
from dotenv import load_dotenv

load_dotenv()

True

In [43]:
from semantic_kernel.functions import kernel_function
from typing import Annotated
import random

class PokemonPlugin:
    @kernel_function(description="Provides a random pokemon name")
    def get_random_pokemon(self) -> Annotated[str, "Returns a random pokemon name"]:
        pokemon = ["Pikachu", "Charizard", "Bulbasaur"]

        return random.choice(pokemon)


In [44]:
from os import getenv
# Endpoint (LLM source)
from openai import AsyncOpenAI
# Service
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

client = AsyncOpenAI(
    api_key= getenv("GITHUB_TOKEN"),
    base_url= getenv("GITHUB_ENDPOINT")
)

service = OpenAIChatCompletion(
    ai_model_id= getenv("GITHUB_MODEL_ID"),
    async_client= client
)

In [45]:
import chromadb
from chromadb.api.models.Collection import Collection

collection = chromadb.PersistentClient(path="./chroma_db").create_collection(
    name="pokemon_documents",
    metadata={"description": "pokemon_service"},
    get_or_create=True,
)

documents = [
    "Pikachu is an electric type pokemon",
    "Charizard is a fire and flying type pokemon",
    "Bulbasaur is a grass and poison type pokemon",
    "Squirtle is a water type pokemon"
]

collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "training", "type": "explanation"} for _ in documents]
)

class rag_plugin:
    def __init__(self, collection: Collection):
        self.collection = collection
    
    @kernel_function(name="get_pokemon_type", description="Retrieves the type of a pokemon from the database")
    def get_pokemon_type(self, query: str):
        rag_results = self.collection.query(
            query_texts=[query],
            include=["documents", "metadatas"],
            n_results=2
        )

        text_rag_result = ""
        if rag_results and rag_results.get("documents") and rag_results.get("metadatas")[0]:
            documents_list = rag_results.get("documents")[0]
            metadata_list = rag_results.get("metadatas")[0]

            for document, metadata in zip(documents_list, metadata_list):
                text_rag_result += f"Document: {document}\nMetadata: {metadata}\n\n"
        else:
            text_rag_result = "No retrieval context found."
        
        return text_rag_result


In [46]:
from pydantic import BaseModel, Field

class Subtask(BaseModel):
    assigned_agent: str = Field(description= "The specific agent assigned to handle this subtask")
    task_details: str = Field(description= "Detailed description of what needs to be done for this subtask")

class PokemonTeamBuildingPlan(BaseModel):
    main_task: str = Field(description="The overall pokemon team build request from the user")
    subtasks: list[Subtask] = Field(description="List of subtasks broken down from the main task, each assigned to a specialized agent")


In [47]:
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.functions import KernelArguments

settings = OpenAIChatPromptExecutionSettings(response_format=PokemonTeamBuildingPlan)

agent = ChatCompletionAgent(
    service= service,
    name= "PokemonAgent",
    instructions= """You are a planner agent for creating competitive pokemon teams.
    Your job is to decide which agents to run based on the user's request.
    Below are the avaliable agents specialised in different tasks:
    - PopularBuilds: For providing information on popular competitve builds on a particular pokemon
    - CompetitionFormats: For providing information on current rules and restrictions on particular competitve pokemon competition formats
    - SmogonTier: For obtaining the tier a pokemon is in, in the Smogon competitve singles format, to determine which tier a team can compete in, and which pokemon are avalaible in the tier
    - StatsInfo: For providing general information on a pokemon's stats, avaliable moves, and other basic pokemon battle information
    - CompetitveMetaInfo: For providing information on the current general metagame and state of a particular competitive format
    - MatchupAgainst: For providing a score, from 1 to 100, with higher numbers indicating a better matchup, on how well a pokemon does against another pokemon one-on-one
    - TeamWith: For providing a score, from 1 to 100, with higher numbers indicating a better matchup, on how well a pokemon complements another pokemon on the same team
    - BuildMaker: For creating a pokemon build, given other potential team members
    - TeamMaker: For creating a final pokemon team, given potential pokemon and given potential builds for each pokemon
    - DefaultAgent: For handling general requests""",
    plugins= [PokemonPlugin(), rag_plugin(collection)],
    arguments= KernelArguments(settings)
)

In [48]:
from pydantic import ValidationError
import json

user_inputs = [
    "Create a competitve pokemon team to use in gen 4 smogon singles using Empoleon",
    "Create a rain team for gen 5 OU"
]

async def main():
    thread = None

    for prompt in user_inputs:
        response = await agent.get_response(messages=prompt, thread=thread)
        thread = response.thread

        try:
            json_response = PokemonTeamBuildingPlan.model_validate(json.loads(response.message.content))
            formatted_json = json_response.model_dump_json(indent=4)
            print(formatted_json)
        except ValidationError as error:
            print(f"Validation error: {str(error)}")
            print(response.content)


        # print(f"\n\n--- User: {prompt}\n", end="", flush=True)
        # first_response = True

        # async for response in agent.invoke_stream(messages=prompt, thread=thread):
        #     thread = response.thread

        #     if first_response:
        #         first_response = False
        #         print(f"--- {response.name}: {response}\n", end="", flush=True)

        #     print(response, end="", flush=True)
    
    if thread:
        await thread.delete()

await main()

{
    "main_task": "Create a competitive Pokemon team to use in Gen 4 Smogon singles with Empoleon",
    "subtasks": [
        {
            "assigned_agent": "SmogonTier",
            "task_details": "Obtain the Smogon tier for Empoleon to confirm its eligibility and available partners."
        },
        {
            "assigned_agent": "PopularBuilds",
            "task_details": "Provide popular competitive builds for Empoleon."
        },
        {
            "assigned_agent": "TeamMaker",
            "task_details": "Create a competitive team including Empoleon and suitable partners in the Gen 4 Smogon tier."
        }
    ]
}
{
    "main_task": "Create a competitive rain team for Gen 5 OU",
    "subtasks": [
        {
            "assigned_agent": "CompetitveMetaInfo",
            "task_details": "Provide information on the current general metagame for Gen 5 OU."
        },
        {
            "assigned_agent": "SmogonTier",
            "task_details": "Obtain the Smogon tier